# Week 2, Day 3 — LangGraph Agents (Gemini API)
**Assignment:** LangGraph — Stateful, Multi-Step & Cyclical Agent Workflows
**Student:** Qasim, BSSE 2022 (2022-SE-49), UET Lahore
**Due:** 9 Sept 2026

This notebook builds a **research-assistant** workflow with LangGraph: it
plans, retrieves, drafts, critiques itself in a self-correction loop,
pauses for human approval before finalizing, and persists its state across
runs with a checkpointer.

**Reproducibility note.** As in the Day 2 notebook, `offline_gemini_llm.build_llm()`
returns a real `ChatGoogleGenerativeAI("gemini-2.5-flash")` when
`GEMINI_API_KEY` is set in the environment, and otherwise falls back to a
small scripted `OfflineGeminiLLM` that implements the exact same
`.invoke(prompt) -> AIMessage` interface. Every graph, node, edge,
conditional route, interrupt, and checkpoint below executes for real — only
the text-generation step inside `planner`/`drafter`/`critique` is scripted
so the notebook runs end-to-end with no API key. Set `GEMINI_API_KEY` and
re-run for live Gemini reasoning; no other code changes.

## Task 1 — Graph Concepts & State Design

**Core building blocks:**
- **StateGraph** — the workflow graph itself: a container you add nodes and
  edges to, then compile into a runnable graph.
- **Nodes** — plain functions that take the current state and return a
  partial update to it (a dict of the fields they changed). Each node is
  one step of the workflow (plan, retrieve, draft, critique, ...).
- **Edges** — fixed transitions from one node to the next (`add_edge`).
- **Conditional edges** — edges whose destination is decided at runtime by
  a router function that inspects the current state (`add_conditional_edges`)
  — this is what makes loops and branching possible.
- **State** — a shared, typed data structure (here a `TypedDict`) that is
  threaded through every node; each node reads what it needs and returns
  only the fields it updates.

### State schema

A research assistant that plans, retrieves, drafts, critiques, and revises
an answer needs to track the query, the plan, retrieved documents, the
current draft, the latest critique, a retry counter, a human-approval flag,
and the final answer.

In [1]:
from typing import TypedDict, List, Optional


class ResearchState(TypedDict):
    query: str
    plan: List[str]
    documents: List[str]
    draft: str
    critique: str
    retries: int
    approved: Optional[bool]
    final_answer: str

### Graph sketch (before coding)

```
START -> planner -> retriever -> drafter -> critique
                                                ├── (bad, retries < max)  -> drafter        [self-correction loop]
                                                └── (good OR retries maxed) -> approval_gate
                                                                                  ├── (approved) -> finalizer -> END
                                                                                  └── (rejected) -> drafter    [loop again]
```

**One deliberate deviation from the assignment's literal Task 4 snippet,
explained up front:** the assignment's example pauses with
`interrupt_before=["finalizer"]`. I pause with `interrupt_before=["approval_gate"]`
instead. Reasoning: by the time a graph is *about* to run `finalizer`, the
conditional-edge router has already decided `finalizer` is next — there is
no node left whose routing can still react to a freshly-set `approved`
flag, so a "reject → loop back to drafter" outcome isn't expressible that
way. Pausing before `approval_gate` instead means: the human sets
`approved` during the pause, `approval_gate` runs, and *its* conditional
edge (evaluated after the pause) reads the fresh flag and routes to
`finalizer` or back to `drafter` correctly. Task 4 below demonstrates both
outcomes.

## Task 2 — Build a Linear Graph

A 4-node backbone: `planner -> retriever -> drafter -> critique`
(critique's conditional routing is added in Task 3). `retriever` reads
from `knowledge_base.py`, a small local document store — the same "plain
Python tool function reading a small local data source" pattern as Day 2's
`lookup_product_price`, applied to this assignment's research domain.
Each node prints state before/after so updates are visible.

In [2]:
import os

from offline_gemini_llm import build_llm
from knowledge_base import retrieve

llm = build_llm(model_name="gemini-3.5-flash")
MAX_RETRIES = 3

c:\Users\p c\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\auth\transport\grpc.py:44: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In October 2026, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


In [3]:
def planner_node(state: ResearchState):
    print("BEFORE planner:", {"query": state["query"]})
    prompt = f"Create a short research plan (3 steps) for answering the question: {state['query']}"
    resp = llm.invoke(prompt)
    plan = [line.strip() for line in resp.content.splitlines() if line.strip()]
    print("AFTER planner:", plan)
    return {"plan": plan}


def retriever_node(state: ResearchState):
    print("BEFORE retriever:", {"plan": state["plan"]})
    docs = retrieve(state["query"], k=4)
    doc_lines = [f"[{d['id']}] {d['title']}: {d['content']}" for d in docs]
    print("AFTER retriever:", [d["id"] for d in docs])
    return {"documents": doc_lines}


def drafter_node(state: ResearchState):
    print("BEFORE drafter:", {"retries": state.get("retries", 0)})
    docs_text = "\n".join(state["documents"])
    if state.get("critique"):
        prompt = (
            f"Revise the draft based on this critique: {state['critique']}\n"
            f"Previous draft: {state['draft']}\n"
            f"Documents:\n{docs_text}"
        )
    else:
        prompt = f"Write a draft answer to '{state['query']}' using ONLY these documents:\n{docs_text}"
    resp = llm.invoke(prompt)
    print("AFTER drafter:", resp.content[:90], "...")
    return {"draft": resp.content}


def critique_node(state: ResearchState):
    retries = state.get("retries", 0) + 1
    print(f"[Loop pass #{retries}] BEFORE critique:", {"draft_len": len(state["draft"])})
    prompt = (
        f"Critique the following draft answer for the question '{state['query']}' "
        f"(attempt {retries}). Draft:\n{state['draft']}"
    )
    resp = llm.invoke(prompt)
    print("AFTER critique:", resp.content[:90], "...")
    return {"critique": resp.content, "retries": retries}


def formatter_node(state: ResearchState):
    print("AFTER formatter.")
    return {"final_answer": f"FORMATTED ANSWER:\n{state['draft']}"}

In [4]:
from langgraph.graph import StateGraph, START, END

linear_workflow = StateGraph(ResearchState)
linear_workflow.add_node("planner", planner_node)
linear_workflow.add_node("retriever", retriever_node)
linear_workflow.add_node("drafter", drafter_node)
linear_workflow.add_node("formatter", formatter_node)

linear_workflow.add_edge(START, "planner")
linear_workflow.add_edge("planner", "retriever")
linear_workflow.add_edge("retriever", "drafter")
linear_workflow.add_edge("drafter", "formatter")
linear_workflow.add_edge("formatter", END)

linear_graph = linear_workflow.compile()
linear_result = linear_graph.invoke({"query": "What are the benefits of RAG?"})
print("\nFINAL STATE KEYS:", list(linear_result.keys()))
print("FINAL ANSWER:\n", linear_result["final_answer"])

BEFORE planner: {'query': 'What are the benefits of RAG?'}
AFTER planner: ['Here is a 3-step research plan to answer the question: **What are the benefits of RAG (Retrieval-Augmented Generation)?**', '### **Step 1: Gather Foundational & Technical Literature**', '*   **Objective:** Understand the core theoretical benefits of RAG as defined by its creators and early adopters.', '*   **Action:**', '*   Review the seminal 2020 Facebook AI Research (FAIR) paper: *"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks."*', '*   Consult documentation from leading AI orchestration frameworks (e.g., LangChain, LlamaIndex) and cloud providers (e.g., AWS, Microsoft Azure) to identify the primary technical advantages they highlight.', '*   **Key focus:** Look for foundational benefits like hallucination reduction, access to external/dynamic data, and source traceability.', '### **Step 2: Conduct a Comparative Analysis**', '*   **Objective:** Identify the unique advantages of RAG by comp

This linear version runs `drafter` exactly once — no self-correction
yet. That's what Task 3 adds.

## Task 3 — Add Conditional Edges & Cycles

Replace the fixed `drafter -> formatter` edge with `drafter -> critique`,
then a **conditional edge** out of `critique` that either loops back to
`drafter` (self-correction) or moves on, guarded by `retries >= MAX_RETRIES`
so the loop can't run forever.

In [5]:
def should_revise(state: ResearchState) -> str:
    if state["retries"] >= MAX_RETRIES:
        return "finish"
    if state["critique"].lower().startswith("good"):
        return "finish"
    return "revise"

### Why this loop-back pattern is awkward in `AgentExecutor` but natural in LangGraph

`AgentExecutor` runs a single implicit loop (call model → maybe call a tool
→ repeat) that only terminates when the model itself stops requesting tool
calls — there's no first-class way to say "run node B, inspect its output
against a threshold, and deterministically send control back to node A a
bounded number of times." You'd have to fake it by writing a custom tool
that *is* the critique step and hope the model chooses to call it and obeys
your termination instructions. In LangGraph, `drafter -> critique -> drafter`
is just a graph with a cycle and an explicit, code-guaranteed exit
condition (`retries >= MAX_RETRIES`) — the control flow is data you can
read and test, not a behavior you're hoping the model exhibits.

## Task 4 — Human-in-the-Loop & Interrupts

`approval_gate` is a real node; `finalizer` is the "risky" action (it
produces the answer that would actually be sent/shown to an end user).
The graph is compiled with `interrupt_before=["approval_gate"]` (see the
Task 1 note above for why), and two runs below demonstrate both outcomes on
independent `thread_id`s: straightforward approval, and reject → another
revision pass → approve.

In [6]:
def approval_gate_node(state: ResearchState):
    print("Human decision received: approved =", state.get("approved"))
    return {}


def route_from_approval(state: ResearchState) -> str:
    return "finalize" if state.get("approved") else "rejected"


def finalizer_node(state: ResearchState):
    print("AFTER finalizer.")
    return {"final_answer": f"FINAL ANSWER (human-approved):\n{state['draft']}"}

In [7]:
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(ResearchState)
workflow.add_node("planner", planner_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("drafter", drafter_node)
workflow.add_node("critique", critique_node)
workflow.add_node("approval_gate", approval_gate_node)
workflow.add_node("finalizer", finalizer_node)

workflow.add_edge(START, "planner")
workflow.add_edge("planner", "retriever")
workflow.add_edge("retriever", "drafter")
workflow.add_edge("drafter", "critique")
workflow.add_conditional_edges("critique", should_revise, {"revise": "drafter", "finish": "approval_gate"})
workflow.add_conditional_edges("approval_gate", route_from_approval, {"finalize": "finalizer", "rejected": "drafter"})
workflow.add_edge("finalizer", END)

checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer, interrupt_before=["approval_gate"])

In [8]:
print("=" * 25, "RUN 1: approval flow", "=" * 25)
config1 = {"configurable": {"thread_id": "approve_thread"}}
graph.invoke({"query": "What are the benefits of RAG?"}, config1)
print("\nPAUSED before:", graph.get_state(config1).next)

# Simulate human approval
graph.update_state(config1, {"approved": True})
final1 = graph.invoke(None, config1)
print("\nFINAL:\n", final1["final_answer"])

========================= RUN 1: approval flow =========================
BEFORE planner: {'query': 'What are the benefits of RAG?'}
AFTER planner: ['Here is a concise 3-step research plan to answer the question: **What are the benefits of RAG (Retrieval-Augmented Generation)?**', '### **Step 1: Literature Review & Source Gathering**', '*   **Objective:** Collect authoritative academic papers, industry whitepapers, and technical documentation defining RAG.', '*   **Action:** Search databases (like Google Scholar or arXiv) and tech blogs (e.g., Pinecone, LangChain, AWS, Cohere) using keywords like "Retrieval-Augmented Generation benefits," "RAG vs. Fine-tuning," and "RAG mitigation of hallucinations."', '*   **Focus:** Identify the core technical problems RAG was designed to solve.', '### **Step 2: Categorization & Comparative Analysis**', '*   **Objective:** Structure the gathered benefits into distinct, actionable categories and compare RAG to alternative LLM optimization methods.', '*

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash
Please retry in 38.612531189s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
}
, retry_delay {
  seconds: 38
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/

AFTER critique: This is an **excellent, highly professional, and technically accurate** draft. It is well- ...
BEFORE drafter: {'retries': 2}
AFTER drafter: Based on the provided documents, the benefits of Retrieval-Augmented Generation (RAG) incl ...
[Loop pass #3] BEFORE critique: {'draft_len': 1509}
AFTER critique: This is an **excellent, highly disciplined draft**. It is well-structured, professional, a ...

PAUSED before: ('approval_gate',)
Human decision received: approved = True
AFTER finalizer.

FINAL:
 FINAL ANSWER (human-approved):
Based on the provided documents, the benefits of Retrieval-Augmented Generation (RAG) include:

*   **Grounded Generation with Fewer Hallucinations:** Instead of relying solely on parameters learned during training, RAG retrieves relevant documents from an external knowledge source so the generator (LLM) can condition its answers directly on those retrieved documents `[doc1]`. Because the model is grounded in this verifiable text at generation time

In [9]:
print("=" * 25, "RUN 2: rejection then approval flow", "=" * 25)
config2 = {"configurable": {"thread_id": "reject_thread"}}
graph.invoke({"query": "What are the benefits of RAG?"}, config2)
print("\nPAUSED before:", graph.get_state(config2).next)

# Simulate a human REJECTING the first time -- routes back to drafter for
# another revision pass instead of finalizing.
graph.update_state(config2, {"approved": False})
graph.invoke(None, config2)
print("\nAfter rejection, paused again before:", graph.get_state(config2).next)

# Human reviews the (already strong) draft again and approves this time.
graph.update_state(config2, {"approved": True})
final2 = graph.invoke(None, config2)
print("\nFINAL after reject-then-approve:\n", final2["final_answer"])

========================= RUN 2: rejection then approval flow =========================
BEFORE planner: {'query': 'What are the benefits of RAG?'}
AFTER planner: ['Here is a concise, 3-step research plan to identify and analyze the benefits of Retrieval-Augmented Generation (RAG):', '### **Step 1: Conduct a Literature Review on Core Technical Benefits**', '* **Action:** Search academic databases (like arXiv) and leading AI research blogs (e.g., Meta AI, Pinecone, LangChain) for foundational papers on RAG.', '* **Focus:** Identify how RAG solves inherent limitations of standard Large Language Models (LLMs). Specifically, look for data on how RAG reduces hallucinations, bypasses knowledge cutoff dates, and provides source verifiability (citations).', '### **Step 2: Perform a Comparative Analysis (RAG vs. Fine-Tuning)**', '* **Action:** Compare RAG against alternative methods of customizing LLMs, specifically fine-tuning and long-context prompt engineering.', '* **Focus:** Analyze the ben

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash
Please retry in 17.481336436s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
}
, retry_delay {
  seconds: 17
}
].


AFTER drafter: Based on the provided documents, the benefits of Retrieval-Augmented Generation (RAG) incl ...
[Loop pass #2] BEFORE critique: {'draft_len': 1205}
AFTER critique: Overall, this is an **excellent, highly professional, and well-structured draft**. It is c ...
BEFORE drafter: {'retries': 2}
AFTER drafter: Based on the provided documents, Retrieval-Augmented Generation (RAG) improves LLM outputs ...
[Loop pass #3] BEFORE critique: {'draft_len': 1449}
AFTER critique: Overall, **Attempt 3 is an excellent, high-quality draft.** It is well-structured, profess ...

PAUSED before: ('approval_gate',)
Human decision received: approved = False
BEFORE drafter: {'retries': 3}
AFTER drafter: Based on the provided documents, here is the revised and polished draft addressing all are ...
[Loop pass #4] BEFORE critique: {'draft_len': 1679}
AFTER critique: This is an excellent, highly polished, and professional draft. It is well-structured, uses ...

After rejection, paused again before: ('a

**When should a real product require human-in-the-loop vs. full
autonomy?** Human review earns its cost when an action is hard or
impossible to undo (sending an email, executing a purchase, deleting data),
when being wrong is costly to someone other than the user driving the
agent, or when the model's confidence/quality signal is itself unreliable
for the task. Full autonomy is reasonable when actions are cheap to undo,
low-stakes, or easily verified after the fact (drafting text a human will
read before it goes anywhere, internal search/summarization, sandboxed
calculations) — the interrupt in this notebook sits right before the one
action (`finalizer`) that represents "this answer is now considered done,"
which is exactly the kind of boundary worth pausing at.

## Task 5 — Persistence & Debugging

`checkpointer=MemorySaver()` (added above) already gives every `thread_id`
a persisted history of state snapshots. `graph.get_state_history(config)`
walks that history newest-first — this is LangGraph's time-travel /
debugging view: you can see exactly which node ran, what the state looked
like, and where execution is paused, at every step of a run.

In [10]:
print("State history for the approval-flow thread (newest first):\n")
for snap in graph.get_state_history(config1):
    print(f"next={snap.next!r:20} retries={snap.values.get('retries')!r:6} approved={snap.values.get('approved')!r}")

State history for the approval-flow thread (newest first):

next=()                   retries=3      approved=True
next=('finalizer',)       retries=3      approved=True
next=('approval_gate',)   retries=3      approved=True
next=('approval_gate',)   retries=3      approved=None
next=('critique',)        retries=2      approved=None
next=('drafter',)         retries=2      approved=None
next=('critique',)        retries=1      approved=None
next=('drafter',)         retries=1      approved=None
next=('critique',)        retries=None   approved=None
next=('drafter',)         retries=None   approved=None
next=('retriever',)       retries=None   approved=None
next=('planner',)         retries=None   approved=None
next=('__start__',)       retries=None   approved=None


Resuming a paused conversation is exactly the `graph.invoke(None, config)`
calls used above in Task 4 — passing `None` as input tells LangGraph "don't
start a new run, continue the persisted one for this `thread_id` from where
it left off." Because `approve_thread` and `reject_thread` are separate
thread IDs on the same checkpointer, their histories never interfere with
each other, which is what makes multi-session persistence (e.g. a user
returning to a paused approval the next day) work.

### LangChain `AgentExecutor` vs. LangGraph — when to reach for each

`AgentExecutor` is the right tool for a single-agent, tool-using loop with
no need for branching, cycles, multi-actor coordination, or mid-run human
checkpoints — e.g. Day 2's shopping assistant, where "reason, call a tool,
read the result, repeat until done" is the whole story. Reach for LangGraph
once the workflow needs any of: an explicit self-correction/retry loop with
a guaranteed exit condition, conditional branching between multiple
distinct steps (not just "call a tool or don't"), a pause point where a
human (or another system) must inject a decision before execution
continues, or persisted, resumable, multi-session state — all of which this
notebook's research assistant needed and a plain `AgentExecutor` has no
clean way to express.

## Part B — Graph Diagram

Generated directly from the compiled graph via `draw_mermaid()`, so it's
guaranteed to match the actual nodes/edges/conditional routes above rather
than being hand-drawn and possibly stale.

In [11]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	retriever(retriever)
	drafter(drafter)
	critique(critique)
	approval_gate(approval_gate<hr/><small><em>__interrupt = before</em></small>)
	finalizer(finalizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	approval_gate -. &nbsp;rejected&nbsp; .-> drafter;
	approval_gate -. &nbsp;finalize&nbsp; .-> finalizer;
	critique -. &nbsp;finish&nbsp; .-> approval_gate;
	critique -. &nbsp;revise&nbsp; .-> drafter;
	drafter --> critique;
	planner --> retriever;
	retriever --> drafter;
	finalizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



```mermaid
graph TD;
	__start__([START]):::first
	planner(planner)
	retriever(retriever)
	drafter(drafter)
	critique(critique)
	approval_gate(approval_gate<hr/><small><em>interrupt: before</em></small>)
	finalizer(finalizer)
	__end__([END]):::last
	__start__ --> planner;
	planner --> retriever;
	retriever --> drafter;
	drafter --> critique;
	critique -. bad, retries < max .-> drafter;
	critique -. good OR retries maxed .-> approval_gate;
	approval_gate -. rejected .-> drafter;
	approval_gate -. approved .-> finalizer;
	finalizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc
```

A PNG render of this same diagram is saved alongside this notebook as
`graph_diagram.png`.